## Web scraping
Realizar una ejecución de proceso web scraping para extraer la
información de la población contenida en la tabla del siguiente vínculo y generar al menos un gráfico o tabla con estos datos.

In [1]:
# importacion de librerias
import requests # Para hacer peticiones desde internet

#URL de la API desde donde se obtienen los datos
Api = "https://live-data.jifo.co/425b93dc-c055-477c-b81a-5d4d9a1275f7" 
#Se realiza la petición a la API y se convierten los datos a formato JSON
res = requests.get(Api).json()

#Se selecciona la pestaña 4 del JSON 'data' pues es la única que contiene información
dataa = res['data'][4] 

#Se crea la sesión de Spark
#SparkSession es necesaria para trabajar con PySpark
spark = SparkSession.builder.appName("BonoPoblacion").getOrCreate()

#Se crea un DataFrame de PySpark: dataa[1:] contiene los datos y dataa[0] contiene los nombres de las columnas
df_spark = spark.createDataFrame(dataa[1:], schema=dataa[0])


NameError: name 'SparkSession' is not defined

In [ ]:

# Crear DataFrame desde dataa
df = pd.DataFrame(dataa[1:], columns=dataa[0])
# Renombrar columnas (la primera viene vacía)
df.columns = ["Fecha", "Positivas", "Total"]
# Limpiar datos numéricos (maneja vacíos y comas)
df["Positivas"] = pd.to_numeric(
    df["Positivas"].astype(str).str.replace(",", ""),
    errors="coerce"
)
df["Total"] = pd.to_numeric(
    df["Total"].astype(str).str.replace(",", ""),
    errors="coerce"
)
#Convertir fechas
df["Fecha"] = pd.to_datetime(df["Fecha"], dayfirst=True, errors="coerce")
#Crear tasa de positividad: Esta es una medida de la relación entre los resultados positivos y el total
df["Tasa_Positividad"] = df["Positivas"] / df["Total"]
# Ordenar por fecha
df = df.sort_values("Fecha")
# Mostrar primeras filas
df.head()

In [ ]:
df_mes = df.resample("M", on="Fecha").sum().reset_index()

df_mes["Tasa_Positividad"] = df_mes["Positivas"] / df_mes["Total"]

df_mes.head()

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df_mes["Fecha"], df_mes["Positivas"], marker="o")

plt.title("Casos Positivos por Mes")
plt.xlabel("Fecha")
plt.ylabel("Casos Positivos")
plt.ticklabel_format(style='plain', axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df_mes["Fecha"], df_mes["Tasa_Positividad"], marker="o")

plt.title("Tasa de Positividad por Mes")
plt.xlabel("Fecha")
plt.ylabel("Tasa")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df_mes = df.resample("ME", on="Fecha").sum().reset_index()

plt.figure(figsize=(12,6))

plt.plot(df_mes["Fecha"], df_mes["Positivas"], marker="o", label="Positivas")
plt.plot(df_mes["Fecha"], df_mes["Total"], marker="o", label="Totales")

plt.title("Casos Positivos vs Total de Pruebas por Mes")
plt.xlabel("Fecha")
plt.ylabel("Cantidad")

plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.ticklabel_format(style='plain', axis='y')
plt.show()